# RAG를 위한 PREPROCESS1 

%pip install sentence-transformers

In [ ]:
# 허깅 페이스 모델로 임베딩
import os
import warnings
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

# ./cache/ 경로에 다운로드 받도록 설정
os.environ["HF_HOME"] = "./cache/" # 즉, 현재 프로젝트 폴더 안에 캐시 폴더에 저장하겠다

# 용량이 작아서 small 로 대체
model_name = "intfloat/multilingual-e5-small"

hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={"device": "cpu"},  # cuda, cpu
    encode_kwargs={"normalize_embeddings": True},
)

# Document Embedding 수행
embedded_query = hf_embeddings.embed_query("LLM에 대해서 알려주세요.")
print(embedded_query)

- `gemini-embedding-2-preview`: LangChain에서 Gemini 임베딩에 사용할 수 있는 최신 계열 모델 예시입니다. 텍스트 의미를 벡터로 변환해 검색, 군집화, RAG 등에 활용할 수 있습니다.
- Gemini 임베딩도 문장/문서의 의미를 고차원 벡터로 바꿔 주며, 벡터 저장소와 함께 사용할 수 있습니다.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError(".env 파일에 GEMINI_API_KEY를 설정하세요.")

os.environ["GOOGLE_API_KEY"] = api_key

gemini_embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [ ]:
vector = gemini_embeddings.embed_query('나는 박찬규입니다')

print(vector)
print(len(vector))

출력예시:

```python
[0.012767945, -0.0025837892, -0.0052350434.....,-0.006252779, 0.0036285543]
```


LangChain에서 Document 객체는 텍스트 기반 데이터를 처리하고 관리하기 위한 기본 단위이다. 

이는 대규모 언어 모델(LLM) 기반 애플리케이션, 특히 검색 증강 생성(RAG) 시스템에서 외부 지식이나 데이터를 표현하는 데 사용되는 핵심 구조입니다.

일반적으로 하나의 Document 객체는 하나의 텍스트 조각(예: 문서의 섹션, PDF 페이지, 웹 페이지의 문단 등)을 나타냅니다.

## Loader란?

document 객체를 랭체인에서 불러올 수 있는데, 이때 문서를 업로드 하는 하나의 코드이다. 

In [ ]:
from langchain_core.documents import Document

# page_content (내용)
# 설명: Document의 실제 텍스트 콘텐츠를 담고 있는 문자열입니다. LLM이 읽고 처리하게 될 핵심 정보입니다.
# 예시: "LangChain은 LLM 기반 애플리케이션 개발을 위한 프레임워크입니다."

# metadata (메타데이터)
# 설명: page_content에 대한 추가적인 정보를 담고 있는 딕셔너리입니다. 이 정보는 Document의 출처, 생성일, 페이지 번호 등 구조적 정보를 포함하여 데이터 검색 및 관리에 유용합니다.
# 예시: {'source': 'guide.pdf', 'page': 5, 'category': 'Tutorial'}

document = Document(page_content="안녕하세요 이건 랭체인의 도큐먼드 입니다")
document.__dict__

# 출력예시: {'id': None,
 'metadata': {},
 'page_content': '안녕하세요? 이건 랭체인의 도큐먼드 입니다',
 'type': 'Document'}

In [ ]:
document.metadata["source"] = "웹페이지"
document.metadata["page"] = 10
document.metadata["author"] = "박찬규"

# 도큐먼트의 속성 확인
document.metadata

# 출력예시: {'source': '웹페이지', 'page': 10, 'author': '박찬규'}

In [ ]:
# Document 객체 생성
my_document = Document(
    page_content="생성형 AI는 텍스트, 이미지, 코드 등 다양한 콘텐츠를 만들 수 있는 기술입니다.",
    metadata={
        "source": "AI_Report_2025.docx",
        "author": "Tech Analyst Team",
        "page_number": 3
    }
)

# 내용과 메타데이터 접근
print(f"--- Document 내용 ---")
print(my_document.page_content)

print(f"\n--- Document 메타데이터 ---")
print(my_document.metadata)

# 특정 메타데이터 값 접근
print(f"소스 파일: {my_document.metadata['source']}")

In [ ]:
import bs4
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_community.document_loaders.csv_loader import UnstructuredCSVLoader
from langchain_community.document_loaders import DataFrameLoader
from langchain_community.document_loaders import UnstructuredExcelLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import JSONLoader

#  - PyPDFLoader: PDF 파일을 로드하는 로더입니다.
#  - CSVLoader: CSV 파일을 로드하는 로더입니다.
#  - UnstructuredHTMLLoader: HTML 파일을 로드하는 로더입니다.
#  - JSONLoader: JSON 파일을 로드하는 로더입니다.
#  - TextLoader: 텍스트 파일을 로드하는 로더입니다.
#  - DirectoryLoader: 디렉토리를 로드하는 로더입니다.

# 랭체인에는 많은 로더가 있는데, 이 로더들은 다 doc 객체를 기반으로 되어있다. 내가 어떤 문서를 다양한 로더로 가져올 수 있다. 
# 로더를 통해서 특정 경로의 PDF를 가져올 수 있는 식이다. 

FILE_PATH = "./SPRi AI Brief_6월호_산업동향_F.pdf"

loader = PyPDFLoader(FILE_PATH)
docs = loader.load()
# 이 코드는 PDF 파일을 읽어서 Langchain에서 다룰 수 있는 문서 객체 리스트로 변환한다. 
# 이 코드를 실행시키면? -> PDF 파일을 실제로 읽고 보통은 페이지 단위로 쪼개서 Document 객체 리스트로 반환한다. 

In [ ]:
print(docs[5].page_content)
print(docs[5].metadata)
# 굉장히 많은 로더들이 있는데, 그 로더들도 이렇게 doc 객체를 불러온다. 

  예를 들어 PDF가 3페이지면:

  docs

  는 대략 이런 구조가 됩니다.
```python
  [
      Document(page_content="1페이지 텍스트...", metadata={"source": "...", "page": 0}),
      Document(page_content="2페이지 텍스트...", metadata={"source": "...", "page": 1}),
      Document(page_content="3페이지 텍스트...", metadata={"source": "...", "page": 2})
  ]
  ```

여기서 content와 metadata를 확인할 수 있다. 